# Star Schema — Customer Support Ticket Analytics

This notebook takes the cleaned dataset (`support-tickets-clean.csv`, output of the Data Quality Analysis notebook) and builds a dimensional (star) model out of it: one fact table plus four dimension tables, each exported as its own CSV so it can be loaded into an SQL database.

**Grain of `FACT_TICKETS`: one row per support ticket.**

| Table | Type | Grain / Contents |
|---|---|---|
| `FACT_TICKETS` | Fact | One row per ticket; FKs to every dimension + all numeric/flag measures |
| `DIM_DATE` | Dimension | One row per calendar day; used twice (role-playing) for opened & resolved |
| `DIM_CHANNEL` | Dimension | One row per channel (chat, email, in-app, phone) |
| `DIM_QUEUE` | Dimension | One row per queue (account, billing, integrations, onboarding, technical) |
| `DIM_PRIORITY` | Dimension | One row per priority (low, normal, high, urgent) |



## 0. Setup

In [ ]:
import pandas as pd
import numpy as np

FILE_NAME = "support-tickets-clean.csv"

try:
    df = pd.read_csv(FILE_NAME, parse_dates=["opened_at", "resolved_at"])
except FileNotFoundError:
    from google.colab import files
    uploaded = files.upload()          # choose support-tickets-clean.csv
    FILE_NAME = list(uploaded.keys())[0]
    df = pd.read_csv(FILE_NAME, parse_dates=["opened_at", "resolved_at"])

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")
df.head(3)

Loaded 4,000 rows x 16 columns


,ticket_id,opened_at,channel,queue,priority,subject,first_response_min,resolved_at,reopened,csat,resolution_min,is_open,has_csat,frt_outlier,resolution_outlier,opened_month
0,TK62871,2023-01-01 02:15:00,in-app,onboarding,high,Error when log in after password reset,63.0,2023-01-01 07:46:35,True,NaN,331.583333,False,False,False,False,2023-01
1,TK60343,2023-01-01 05:41:00,email,technical,normal,Question about two-factor authentication,7.0,2023-01-01 09:07:58,False,NaN,206.966667,False,False,False,False,2023-01
2,TK62950,2023-01-01 06:05:00,in-app,billing,urgent,Unable to connecting the Snowflake sync,44.0,2023-01-01 08:14:24,False,NaN,129.400000,False,False,False,False,2023-01


## 1. DIM_DATE (role-playing dimension)

Built as a **continuous calendar** spanning the earliest to the latest date seen across *either* `opened_at` or `resolved_at` — not just the dates that happen to appear in the data. A continuous calendar lets the dashboard show days with zero tickets (e.g. for a "tickets per day" trend line) instead of silently skipping them.

In [ ]:
all_dates = pd.concat([df["opened_at"].dt.normalize(), df["resolved_at"].dt.normalize()]).dropna()

dim_date = pd.DataFrame({"full_date": pd.date_range(all_dates.min(), all_dates.max(), freq="D")})
dim_date["date_key"]     = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]         = dim_date["full_date"].dt.year
dim_date["quarter"]      = dim_date["full_date"].dt.quarter
dim_date["month"]        = dim_date["full_date"].dt.month
dim_date["month_name"]   = dim_date["full_date"].dt.month_name()
dim_date["day"]          = dim_date["full_date"].dt.day
dim_date["day_of_week"]  = dim_date["full_date"].dt.dayofweek + 1   # 1=Mon ... 7=Sun
dim_date["day_name"]     = dim_date["full_date"].dt.day_name()
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)
dim_date["is_weekend"]   = dim_date["day_of_week"].isin([6, 7])

dim_date = dim_date[["date_key", "full_date", "year", "quarter", "month", "month_name",
                      "day", "day_of_week", "day_name", "week_of_year", "is_weekend"]]

print(f"DIM_DATE: {len(dim_date):,} rows, {dim_date['full_date'].min().date()} -> {dim_date['full_date'].max().date()}")
dim_date.head(3)

DIM_DATE: 400 rows, 2023-01-01 -> 2024-02-04


,date_key,full_date,year,quarter,month,month_name,day,day_of_week,day_name,week_of_year,is_weekend
0,20230101,2023-01-01,2023,1,1,January,1,7,Sunday,52,True
1,20230102,2023-01-02,2023,1,1,January,2,1,Monday,1,False
2,20230103,2023-01-03,2023,1,1,January,3,2,Tuesday,1,False


## 2. DIM_CHANNEL, DIM_QUEUE, DIM_PRIORITY

Small, low-cardinality lookup dimensions with simple surrogate keys. `priority` gets an explicit `priority_rank` (1=low … 4=urgent) since alphabetical order would otherwise scramble the natural severity order in any BI tool sort.

In [ ]:
dim_channel = pd.DataFrame({"channel_name": sorted(df["channel"].unique())})
dim_channel.insert(0, "channel_key", range(1, len(dim_channel) + 1))

dim_queue = pd.DataFrame({"queue_name": sorted(df["queue"].unique())})
dim_queue.insert(0, "queue_key", range(1, len(dim_queue) + 1))

priority_order = ["low", "normal", "high", "urgent"]
assert set(priority_order) == set(df["priority"].unique()), "Unexpected priority value found"
dim_priority = pd.DataFrame({
    "priority_name": priority_order,
    "priority_rank": range(1, len(priority_order) + 1),
})
dim_priority.insert(0, "priority_key", range(1, len(dim_priority) + 1))

print("DIM_CHANNEL"); display(dim_channel)
print("DIM_QUEUE"); display(dim_queue)
print("DIM_PRIORITY"); display(dim_priority)

DIM_CHANNEL


,channel_key,channel_name
0,1,chat
1,2,email
2,3,in-app
3,4,phone


DIM_QUEUE


,queue_key,queue_name
0,1,account
1,2,billing
2,3,integrations
3,4,onboarding
4,5,technical


DIM_PRIORITY


,priority_key,priority_name,priority_rank
0,1,low,1
1,2,normal,2
2,3,high,3
3,4,urgent,4


## 3. FACT_TICKETS

Join the cleaned ticket log to each dimension's surrogate key, derive the two date keys, and keep every measure and quality flag produced by the Data Quality notebook (`is_open`, `has_csat`, `frt_outlier`, `resolution_outlier`).

In [ ]:
fact = df.merge(dim_channel, left_on="channel", right_on="channel_name", how="left")
fact = fact.merge(dim_queue, left_on="queue", right_on="queue_name", how="left")
fact = fact.merge(dim_priority[["priority_key", "priority_name"]], left_on="priority", right_on="priority_name", how="left")

fact["date_opened_key"] = fact["opened_at"].dt.strftime("%Y%m%d").astype(int)
fact["date_resolved_key"] = fact["resolved_at"].dt.strftime("%Y%m%d")
fact["date_resolved_key"] = pd.to_numeric(fact["date_resolved_key"], errors="coerce").astype("Int64")

fact["ticket_count"] = 1
fact = fact.rename(columns={"csat": "csat_score"})

fact_tickets = fact[[
    "ticket_id", "date_opened_key", "date_resolved_key",
    "channel_key", "queue_key", "priority_key",
    "opened_at", "resolved_at", "subject",
    "first_response_min", "resolution_min", "csat_score",
    "is_open", "has_csat", "reopened", "frt_outlier", "resolution_outlier",
    "ticket_count",
]].copy()

print(f"FACT_TICKETS: {len(fact_tickets):,} rows")
fact_tickets.head(3)

FACT_TICKETS: 4,000 rows


,ticket_id,date_opened_key,date_resolved_key,channel_key,queue_key,priority_key,opened_at,resolved_at,subject,first_response_min,resolution_min,csat_score,is_open,has_csat,reopened,frt_outlier,resolution_outlier,ticket_count
0,TK62871,20230101,20230101,3,4,3,2023-01-01 02:15:00,2023-01-01 07:46:35,Error when log in after password reset,63.0,331.583333,NaN,False,False,True,False,False,1
1,TK60343,20230101,20230101,2,5,2,2023-01-01 05:41:00,2023-01-01 09:07:58,Question about two-factor authentication,7.0,206.966667,NaN,False,False,False,False,False,1
2,TK62950,20230101,20230101,3,2,4,2023-01-01 06:05:00,2023-01-01 08:14:24,Unable to connecting the Snowflake sync,44.0,129.400000,NaN,False,False,False,False,False,1


## 4. Referential Integrity Checks

Before exporting, confirm every fact row has a valid (non-null) key for every dimension it should always have, and that `date_resolved_key` is null exactly for open tickets (never for closed ones).

In [ ]:
checks = {
    "ticket_id not null":           fact_tickets["ticket_id"].isna().sum(),
    "date_opened_key not null":     fact_tickets["date_opened_key"].isna().sum(),
    "channel_key not null":         fact_tickets["channel_key"].isna().sum(),
    "queue_key not null":           fact_tickets["queue_key"].isna().sum(),
    "priority_key not null":        fact_tickets["priority_key"].isna().sum(),
    "ticket_id duplicated":         fact_tickets["ticket_id"].duplicated().sum(),
}

for check, n_failed in checks.items():
    status = "PASS" if n_failed == 0 else f"FAIL ({n_failed})"
    print(f"{check:32s}: {status}")

# date_resolved_key should be null iff the ticket is still open
mismatch = (fact_tickets["date_resolved_key"].isna()) != (fact_tickets["is_open"])
print(f"{'is_open matches date_resolved_key':32s}: {'PASS' if mismatch.sum() == 0 else f'FAIL ({mismatch.sum()})'}")

# every date_opened_key / date_resolved_key must exist in DIM_DATE
valid_keys = set(dim_date["date_key"])
orphan_opened = ~fact_tickets["date_opened_key"].isin(valid_keys)
orphan_resolved = fact_tickets["date_resolved_key"].notna() & ~fact_tickets["date_resolved_key"].isin(valid_keys)
print(f"{'date_opened_key found in DIM_DATE':32s}: {'PASS' if orphan_opened.sum() == 0 else f'FAIL ({orphan_opened.sum()})'}")
print(f"{'date_resolved_key found in DIM_DATE':32s}: {'PASS' if orphan_resolved.sum() == 0 else f'FAIL ({orphan_resolved.sum()})'}")

ticket_id not null              : PASS
date_opened_key not null        : PASS
channel_key not null            : PASS
queue_key not null              : PASS
priority_key not null           : PASS
ticket_id duplicated            : PASS
is_open matches date_resolved_key: PASS
date_opened_key found in DIM_DATE: PASS
date_resolved_key found in DIM_DATE: PASS


## 5. Export

Write each table to its own CSV — ready to load into Power BI / Tableau (as separate tables joined on the key columns) or `COPY`/`LOAD DATA` into the tables created by `star_schema.sql`.

In [ ]:
dim_date.to_csv("dim_date.csv", index=False)
dim_channel.to_csv("dim_channel.csv", index=False)
dim_queue.to_csv("dim_queue.csv", index=False)
dim_priority.to_csv("dim_priority.csv", index=False)
fact_tickets.to_csv("fact_tickets.csv", index=False)

print("Saved: dim_date.csv, dim_channel.csv, dim_queue.csv, dim_priority.csv, fact_tickets.csv")
print()
print(f"{'Table':16s}{'Rows':>8s}")
for name, t in [("dim_date", dim_date), ("dim_channel", dim_channel),
                ("dim_queue", dim_queue), ("dim_priority", dim_priority),
                ("fact_tickets", fact_tickets)]:
    print(f"{name:16s}{len(t):>8,}")

Saved: dim_date.csv, dim_channel.csv, dim_queue.csv, dim_priority.csv, fact_tickets.csv

Table               Rows
dim_date             400
dim_channel            4
dim_queue              5
dim_priority           4
fact_tickets       4,000
